In [ ]:
# Symbolic Curvature Problem Solvers
# This notebook demonstrates the use of symbolic field-driven heuristics to solve:
# - Traveling Salesman Problem (TSP)
# - Graph Coloring
# - Boolean Satisfiability Problem (SAT)

import numpy as np
import pandas as pd
import networkx as nx
import random
from itertools import product
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt

# Utility Functions

def generate_random_3sat(num_vars=6, num_clauses=20):
    cnf = []
    for _ in range(num_clauses):
        clause = set()
        while len(clause) < 3:
            var = random.randint(1, num_vars)
            sign = random.choice([-1, 1])
            clause.add(sign * var)
        cnf.append(list(clause))
    return cnf

def symbolic_sat_solver(cnf, symbolic_field, entropy_field):
    best_score = float('-inf')
    best_assignment = None
    satisfiable_found = False

    for assignment in product([False, True], repeat=len(symbolic_field)):
        score = sum(symbolic_field[i] if val else -entropy_field[i] for i, val in enumerate(assignment))
        satisfies = True
        for clause in cnf:
            if not any((assignment[abs(lit) - 1] if lit > 0 else not assignment[abs(lit) - 1]) for lit in clause):
                satisfies = False
                break
        if satisfies:
            satisfiable_found = True
            if score > best_score:
                best_score = score
                best_assignment = assignment
    return best_assignment, best_score, satisfiable_found

def generate_random_graph(num_nodes=30, edge_prob=0.15):
    G = nx.Graph()
    G.add_nodes_from(range(num_nodes))
    for i in range(num_nodes):
        for j in range(i + 1, num_nodes):
            if random.random() < edge_prob:
                G.add_edge(i, j)
    return G

def gohst_guided_coloring(G, symbolic_field, entropy_field, field_size):
    color_assignment = {}
    for node in sorted(G.nodes()):
        used = {color_assignment[neighbor] for neighbor in G.neighbors(node) if neighbor in color_assignment}
        x, y = divmod(node, int(field_size ** 0.5))
        x = np.clip(x, 0, field_size - 1)
        y = np.clip(y, 0, field_size - 1)
        symbolic_boost = symbolic_field[x, y]
        entropy_penalty = entropy_field[x, y]
        color_scores = {}
        for c in range(len(G)):
            if c not in used:
                color_scores[c] = symbolic_boost - entropy_penalty
        best_color = min(color_scores, key=lambda k: (-color_scores[k], k)) if color_scores else (max(used) + 1 if used else 0)
        color_assignment[node] = best_color
    return color_assignment

def generate_cities(n, seed=42):
    np.random.seed(seed)
    return np.random.rand(n, 2)

def distance(a, b):
    return np.linalg.norm(a - b)

def total_cost(path, coords):
    return sum(distance(coords[path[i]], coords[path[(i + 1) % len(path)]]) for i in range(len(path)))

def greedy_tsp(coords):
    n = len(coords)
    visited = [False] * n
    path = [0]
    visited[0] = True
    for _ in range(1, n):
        last = path[-1]
        next_city = min((i for i in range(n) if not visited[i]), key=lambda j: distance(coords[last], coords[j]))
        path.append(next_city)
        visited[next_city] = True
    return path

# TSP Example
coords = generate_cities(20)
tsp_path = greedy_tsp(coords)
tsp_cost = total_cost(tsp_path, coords)

# Graph Coloring Example
num_nodes = 50
G = generate_random_graph(num_nodes, edge_prob=0.2)
field_dim = max(50, num_nodes)
symbolic_field = gaussian_filter(np.random.rand(field_dim, field_dim), sigma=2)
entropy_field = gaussian_filter(np.random.rand(field_dim, field_dim), sigma=2)
coloring = gohst_guided_coloring(G, symbolic_field, entropy_field, field_dim)
num_colors_used = len(set(coloring.values()))

# SAT Example
cnf = generate_random_3sat(num_vars=6, num_clauses=20)
symbolic_field = gaussian_filter(np.random.rand(6), sigma=1)
entropy_field = gaussian_filter(np.random.rand(6), sigma=1)
solution, score, satisfiable = symbolic_sat_solver(cnf, symbolic_field, entropy_field)

# Results Summary
print("TSP Path Cost:", tsp_cost)
print("Graph Coloring - Colors Used:", num_colors_used)
print("SAT Satisfiable:", satisfiable)
